# ML Projekt
## Deborah Burri, Doga Kaya, Lea Hanimann, Elisa Sirigu

### 1. Einleitung
Unsere Arbeit beschäftigt sich mit der Vorhersage des Erfolgs von Werbekampagnen mithilfe von Machine-Learning-Verfahren. Die Daten stammen von der Eventagentur Encore Events, welche Kampagnen für Sport-, Kultur- und Konferenzveranstaltungen auf verschiedenen Online-Kanälen wie Google, Facebook und Instagram durchführt.

Unser Ziel besteht darin, bereits während der Laufzeit einer Kampagne vorherzusagen, ob die Kampagne am Ende erfolgreich sein wird. Als erfolgreich gilt eine Kampagne dann, wenn mindestens 90 % der angestrebten Impressions erreicht werden. Durch eine frühe Vorhersage können Kampagnen mit geringem Erfolgspotenzial rechtzeitig erkannt und mit zusätzlichen Massnahmen unterstützt werden.

Für die Umsetzung analysieren und bereiten wir die vorhandenen Kampagnendaten auf und werten diese mit verschiedenen Klassifikationsalgorithmen aus. Anschliessend bewerten wir das geeignetste Modell anhand passender Evaluationsmethoden und interpretieren dessen praktischen Nutzen für Encore Events.

### 2. Daten laden

In [ ]:
import pandas as pd
df_basic = pd.read_csv("inputfiles/campaign_basic_information.csv")
df_channels = pd.read_csv("inputfiles/campaign_channels.csv")
df_snapshots = pd.read_csv("inputfiles/campaign_snapshots.csv")

### 3. Channels und Snapshots pivotieren
#### 3.1 Transformation vom Long-Format ins Wide-Format
Die Tabellen `campaign_channels` und `campaign_snapshots` enthalten Informationen über die einzelnen Kampagnen von Encore Events. Diese liegen ursprünglich im sogenannten Long-Format vor. Dabei kann eine Kampagne mehrere Zeilen besitzen, wenn mehrere Kanäle oder mehrere zeitliche Snapshots vorhanden sind.

Für das Machine-Learning-Modell wird jedoch eine Struktur benötigt, bei der jede Kampagne genau eine Zeile besitzt. Deshalb werden die Daten mithilfe einer Pivot-Transformation in ein Wide-Format umgewandelt.

Durch diese Transformation können alle Informationen einer Kampagne kompakt in einer einzigen Zeile dargestellt und später vom Modell verarbeitet werden.

#### 3.2 Channels
Bei den Channels wird für jeden Marketingkanal eine eigene Spalte erstellt.

Die Werte werden binär codiert:

- `1` = Kanal wurde verwendet
- `0` = Kanal wurde nicht verwendet

Diese Darstellung entspricht einem Multi-Hot-Encoding und ermöglicht es, kategoriale Informationen numerisch abzubilden.

#### 3.3 Snapshots
Bei den Snapshots werden die einzelnen Zeitpunkte (`snapshot_1` bis `snapshot_4`) in separate Spalten überführt.

Dadurch kann die zeitliche Entwicklung einer Kampagne als Feature-Set für das Machine-Learning-Modell verwendet werden.

In [ ]:
df_channels["value"] = 1

channels_pivot = df_channels.pivot_table(
    index="campaign_id",
    columns="channel",
    values="value",
    fill_value=0
).reset_index()

channels_pivot.columns.name = None

In [66]:
snapshots_impr = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="impressions"
)
snapshots_impr.columns = [f"impressions_s{col}" for col in snapshots_impr.columns]

snapshots_days = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="days_after_start"
)
snapshots_days.columns = [f"days_after_start_s{col}" for col in snapshots_days.columns]

snapshots_pivot = snapshots_impr.merge(
    snapshots_days,
    left_index=True,
    right_index=True 
).reset_index()

### 4. Tabellen zusammenführen
An dieser Stelle haben wir die drei ursprünglichen Tabellen (`campaign_basic_information`, `campaign_snapshots` und `campaign_channels`) zu einer Tabelle zusammengeführt, da dies die Weiterarbeit vereinfacht. 

In [67]:
df = df_basic.merge(snapshots_pivot, on="campaign_id", how="left")
df = df.merge(channels_pivot, on="campaign_id", how="left")

### 5. Zielvariable erstellen

Die Zielvariable `needs_intervention` beschreibt, ob eine Kampagne voraussichtlich Unterstützung benötigt.

Sie basiert auf der Spalte der finalen Impressions (4. Snapshot), welche man im pivotierten Snapshot Datensatz finden kann.

Bei `needs_intervention`gilt:
- `1` = Ziel (90%) wird nicht erreicht
- `0` = Ziel wird erreicht

In [ ]:
df["needs_intervention"] = (
    df["impressions_s4"] < 0.9 * df["target"]
).astype(int)

### 6. Fehlende Werte in den Daten
Mithilfe von KI-Tools haben wir die Datensätze analysiert, um fehlende Werte zu identifizieren und deren Anteil zu bestimmen. Dabei haben wir in den folgenden Spalten fehlende Einträge festgestellt:

- Impressions: 17 fehlende Werte von insgesamt 964 Einträgen (1.8%)
- Channels: 393 fehlende Werte von insgesamt 964 Einträgen (40.8%)
- Category: 2 fehlende Werte von insgesamt 241 Einträgen (0.8%)

#### 6.1 Fehlende Channel-Spalten
Nach der Pivotierung entstehen leere Zellen für alle Channels, auf denen eine Kampagne nicht aktiv ist. Da 40.8% der möglichen Channel-Einträge nicht belegt sind, haben wir diese fehlenden Werte mit 0 ersetzt, um NaN-Werte im Datensatz zu vermeiden.

In [68]:
channel_cols = ["facebook", "instagram", "google_search", "google_display"]

for col in channel_cols:
    if col not in df.columns:
        df[col] = 0

df[channel_cols] = df[channel_cols].fillna(0).astype(int)

#### 6.2 Fehlende Impression-Spalten
Um die fehlenden Werte zu ersetzen haben wir uns überlegt, der jeweils fehlende Wert durch Berechnung des Wachstums mit den anderen zwei Werten zu interpretieren. Dafür mussten wir allerdings erst kontrollieren, ob es eine Kampagne gibt bei der mehr als ein Impression-Wert fehlt. Weil das nicht der Fall ist haben wir wie folgt die fehlenden Werte berechnet:
- impressions_s3 fehlt → Trend von S1→S2 wird zu S3 aufaddiert (s3 = s2 + (s2 - s1))
- impressions_s2 fehlt → Mittelwert von S1 und S3 (s2 = (s1 + s3) / 2)
- impressions_s1 fehlt → Trend von S2→S3 wird rückwärts auf S1 angewendet (s1 = s2 - (s3 - s2))

In [ ]:
def impute_missing_impressions(row):
    s1, s2, s3 = row["impressions_s1"], row["impressions_s2"], row["impressions_s3"]

    if pd.isna(s3) and not pd.isna(s1) and not pd.isna(s2):
        s3 = s2 + (s2 - s1)

    elif pd.isna(s2) and not pd.isna(s1) and not pd.isna(s3):
        s2 = (s1 + s3) / 2

    elif pd.isna(s1) and not pd.isna(s2) and not pd.isna(s3):
        s1 = s2 - (s3 - s2)

    row["impressions_s1"] = s1
    row["impressions_s2"] = s2
    row["impressions_s3"] = s3
    return row

df = df.apply(impute_missing_impressions, axis=1) #axis=0 wäre Spaltenweise

#### 6.3 Fehlende Category-Spalten
Weil wir nur sehr wenige fehlende Werte bei der Spalte `category`haben und wir diese Information nicht zum Trainieren des Modells verwenden, haben wir uns entschieden die fehlenden Werte zu ignorieren.

In [ ]:
if "category" in df.columns:
    df = df.drop(columns=["category"])

### 7. Feature Engineering
Nachdem die Daten aufbereitet sind, haben wir als Team diskutiert, welche zusätzlichen Merkmale hilfreich sein könnten, um eine gute Vorhersage zu ermöglichen. Jedes Teammitglied hat eigene Feature-Ideen eingebracht, erklärt und begründet. Das Ziel ist, aus den vorhandenen Rohdaten aussagekräftige Kennzahlen zu generieren, die dem Modell helfen, Muster zwischen gut und schlecht laufenden Kampagnen zu erkennen.

Wichtig dabei: Alle Features basieren ausschliesslich auf Daten, die zum Zeitpunkt des dritten Snapshots verfügbar sind. Informationen aus dem vierten Snapshot werden bewusst ausgeschlossen, um Data Leakage zu vermeiden.

#### 7.1 Zeitfortschritt
`time_progress_s3` ist ein numerisches Merkmal (Wertebereich 0–1). Mit dieser Kennzahl wird der zeitliche Kontext der Kampagne messbar gemacht. Ein hoher Fortschritt bei den Impressions ist nur dann positiv zu bewerten, wenn er in einem angemessenen Verhältnis zur verstrichenen Zeit steht. Durch die Division der bereits vergangenen Tage durch die Gesamtlaufzeit erhalten wir einen Wert zwischen 0 und 1, der ausdrückt, wie viel der verfügbaren Zeit bereits verbraucht ist.
Ein Beispiel: Eine Kampagne mit `time_progress_s3` = 0.85 und einer Zielerreichung von 40% ist trotz absolut hoher Impressions-Zahlen ein klarer Interventionsfall. Diesen Zusammenhang kann das Modell nur erkennen, wenn der zeitliche Kontext explizit als Feature vorliegt.
`remaining_days` ist ebenfalls numerisch und wird als Hilfsgrösse für spätere Berechnungen benötigt.

In [ ]:
df["time_progress_s3"] = df["days_after_start_s3"] / df["days"]
df["remaining_days"] = df["days"] - df["days_after_start_s3"]

#### 7.2 Fortschritt relativ zum Ziel
`impr_s3_ratio` ist ein numerisches Merkmal (Wertebereich typischerweise 0–1). Dieses Feature dient der Normalisierung der Zielerreichung. Da die Kampagnen sehr unterschiedliche Zielgrössen haben, sind absolute Impression-Zahlen nicht direkt vergleichbar. Die Ratio transformiert den aktuellen Stand in eine relative Kennzahl, die unabhängig von der Kampagnengrösse interpretierbar ist.
Ein Wert von 0.7 bedeutet konsistent, dass 70% des Ziels erreicht sind, egal ob das Ziel bei 10'000 oder 5 Millionen Impressions liegt. Das ermöglicht dem Modell, universelle Muster für Erfolg oder Misserfolg zu lernen und Kampagnen verschiedener Grössenordnungen sinnvoll miteinander zu vergleichen.

In [70]:
df["impr_s3_ratio"] = df["impressions_s3"] / df["target"]

#### 7.3 Impressionen pro Tag
`impr_per_day_s3` ist ein numerisches Merkmal und beschreibt die aktuelle Ausspielungsgeschwindigkeit (Velocity). Während `impr_s3_ratio` den bisherigen Status beschreibt, liefert dieses Feature einen Indikator für die zukünftige Entwicklung. Das Modell benötigt diese Information, um zu beurteilen, ob das bisherige Tempo ausreicht, um die restliche Distanz zum Ziel in der verbleibenden Zeit zu überbrücken.
Wenn eine Kampagne gerade erst gestartet ist, ist `days_after_start_s3` = 0. Eine Division durch 0 würde zu einem Fehler oder einem inf-Wert führen. Deshalb wird mit `.replace(0, 1)` die Null temporär durch eine Eins ersetzt, was zu einem stabilen, konservativen Schätzwert führt.

Wie sich die drei Features unterscheiden, zeigt folgendes Beispiel:
- `impr_s3_ratio` = 0.5 → 50% des Ziels erreicht
- `time_progress_s3` = 0.8 → aber schon 80% der Zeit verbraucht → zu langsam
- `impr_per_day_s3` = 10'000 → Velocity hilft dem Modell zu sehen, ob das Ziel noch erreichbar ist

In [72]:
df["impr_per_day_s3"] = df["impressions_s3"] / df["days_after_start_s3"].replace(0, 1)

#### 7.4 Benötigte Performance & Speed Ratio
`required_impr_per_day` und `speed_ratio` sind numerische Merkmale. Dieses Feature beantwortet die zentrale Frage: Ist das aktuelle Tempo ausreichend, um das Ziel noch zu erreichen?
`required_impr_per_day` berechnet, wie viele Impressions pro Tag ab dem dritten Snapshot noch nötig wären, um 90% des Ziels zu erreichen. `speed_ratio` setzt diese benötigte Geschwindigkeit direkt in Relation zur bisherigen Performance:

Interpretation der Ergebnisse:
- `<1.0` Kampagne ist schneller als nötig - kein Handlungsbedarf
- `=1.0` Kampagne liegt perfekt im Plan
- `>1.0` Kampagne ist zu langsam - Intervention möglicherweise nötig (z.B. 1.5 → 50% zu langsam)

`speed_ratio` ist damit eines der direktesten und aussagekräftigsten Features für unsere Zielvariable `needs_intervention`.

Falls in einer vorherigen Rechnung ein unendlicher Wert (inf) entstanden ist, wird dieser mit `.replace([float("inf")], 0)` hart auf 0 zurückgesetzt.

In [73]:
df["remaining_target"] = 0.9 * df["target"] - df["impressions_s3"]

df["required_impr_per_day"] = df["remaining_target"] / df["remaining_days"]
df["required_impr_per_day"] = df["required_impr_per_day"].replace([float("inf")], 0)

df["speed_ratio"] = df["required_impr_per_day"] / df["impr_per_day_s3"].replace(0, 1)

#### 7.5 Prognose
`projected_ratio` ist ein numerisches Merkmal. `projected_total` wird als Zwischenwert berechnet und nach dem Feature Engineering wieder entfernt. Die Berechnung basiert auf der Annahme, dass die bisherige Ausspielgeschwindigkeit konstant bleibt (lineare Hochrechnung). `projected_ratio` vergleicht diesen prognostizierten Endwert mit dem Zielwert:

- `<1.0` bedeutet Unterlieferung
- `=1.0` bedeutet Punktlandung
- `>1.0` bedeutet Überlieferung

Obwohl das Modell diese Berechnung aus anderen Features ableiten könnte, hilft das explizite Bereitstellen als Feature dabei, schneller und robuster zu lernen.

In [ ]:
df["projected_total"] = df["impressions_s3"] + df["impr_per_day_s3"] * df["remaining_days"]
df["projected_ratio"] = df["projected_total"] / df["target"]

#### 7.6 Wachstum
`growth_1_2` und `growth_2_3` sind numerische Merkmale, die zeigen, wie sich die Impressions zwischen den einzelnen Snapshot-Zeitpunkten entwickeln. Der aktuelle Impression-Stand allein sagt nichts darüber aus, ob eine Kampagne an Fahrt gewinnt oder verliert. Eine Kampagne mit 600'000 Impressions könnte auf dem Weg nach oben sein oder kurz vor dem Stillstand.
Ein positiver und zunehmender Trend ist ein gutes Zeichen, ein sinkender oder negativer Trend ein Frühwarnsignal. Besonders wertvoll ist dieses Feature für Kampagnen, die aktuell noch solide aussehen, sich aber in eine ungünstige Richtung entwickeln, das ohne Wachstumsbetrachtung unsichtbar bliebe.

In [74]:
df["growth_1_2"] = df["impressions_s2"] - df["impressions_s1"]
df["growth_2_3"] = df["impressions_s3"] - df["impressions_s2"]

#### 7.7 Wachstumsbeschleunigung
`growth_acceleration` ist ein numerisches Merkmal und misst, ob sich der Wachstumstrend verbessert oder verschlechtert. Konzeptuell entspricht dies der zweiten Ableitung der Impression-Kurve:

- Positiver Wert: Kampagne gewinnt zunehmend an Schwung
- Negativer Wert: Wachstum verlangsamt sich - ein Warnsignal, auch wenn der aktuelle Trend noch positiv ist
- Stark negativer Wert: Kampagne verliert deutlich an Dynamik, dies bedeutet ein hohes Interventionsrisiko

Dieses Feature hilft dem Modell nicht nur den aktuellen Trend, sondern auch dessen Veränderungsrichtung zu erkennen.

In [75]:
df["growth_acceleration"] = df["growth_2_3"] - df["growth_1_2"]

#### 7.8 Budget Effizienz
`budget_per_day` ist ein numerisches Merkmal und beschreibt, wie viel Budget pro Kampagnentag zur Verfügung steht. Dadurch wird das Gesamtbudget in Relation zur Laufzeit gesetzt und vergleichbar gemacht, unabhängig von der Kampagnendauer. Dieses Merkmal ist relevant, da Kampagnen mit höherem täglichem Budget tendenziell eine höhere Reichweite erzielen können und somit eine bessere Erfolgswahrscheinlichkeit haben.

In [ ]:
df["budget_per_day"] = df["budget"] / df["days"]

#### 7.9 Anzahl Channels
`n_channels` ist ein numerisches Merkmal (Wertebereich 0–4). Mit diesem Feature wird erfasst, über wie viele Kanäle eine Kampagne gleichzeitig läuft. Die einzelnen Kanalindikatoren werden dabei zu einer Gesamtanzahl aggregiert. Dies ist wichtig, da eine breitere Streuung über mehrere Kanäle oft mit höherer Sichtbarkeit und besseren Ergebnissen verbunden ist, da unterschiedliche Zielgruppen über verschiedene Plattformen erreicht werden können. Durch die Aggregation zu einer einzelnen Zahl bleibt das Feature einfach interpretierbar, ohne dass das Modell die Kanalstruktur selbst herleiten muss.

In [ ]:
channel_cols = ["facebook", "instagram", "google_search", "google_display"]
df["n_channels"] = df[channel_cols].sum(axis=1)

#### 7.10 Google-Nutzung
`google_any` ist ein binäres Merkmal (0 = kein Google-Kanal, 1 = mindestens ein Google-Kanal aktiv). Die Variable zeigt, ob eine Kampagne mindestens einen Google-Kanal nutzt (Search oder Display). Dieses Merkmal ist aussagekräftig, da die Nutzung von Google-Plattformen häufig mit hoher Reichweite und gezielter Ansprache verbunden ist. Die Einzelkanäle `google_search` und `google_display` bleiben dabei als separate Features erhalten.

In [81]:
df["google_any"] = ((df["google_search"] + df["google_display"]) > 0).astype(int)

### 8. Data Leakage und Redundanz vermeiden
Um die Integrität des Modells sicherzustellen, müssen wir zwei kritische Aspekte beim Bereinigen des Datensatzes beachten. Diese werden in den folgenden zwei Kapitel näher erläutert. 

#### 8.1 Data Leakage
Da `impressions_s4` den Endzustand der Kampagne darstellt, enthält dieses Feature Informationen aus der Zukunft. Würde das Modell mit diesen Daten trainiert, würde es "schummeln", da es das Ergebnis bereits kennt. Dies führt zu unrealistisch guten Ergebnissen im Training, die in der echten Anwendung (Prediction) versagen.

#### 8.2 Redundanz und Fokus
Durch das Feature Engineering haben wir bereits aussagekräftige Kennzahlen (wie Ratios und Geschwindigkeiten) erstellt. Die ursprünglichen Rohdaten tragen keinen weiteren Informationswert bei oder könnten das Modell sogar verwirren.
Unsere Tabelle haben wir auf diese Weise bereinigt, weil:
- Ein Modell darf niemals Zugriff auf Daten haben, die zum Zeitpunkt der Vorhersage noch nicht existieren,
- berechnete Features ersetzen die zugrunde liegenden Rohwerte, um Multikollinearität zu vermeiden,
- nicht-relevante Merkmale wie IDs oder Zeitstempel werden entfernt, um das Rauschen im Datensatz zu minimieren und
- bei einem Unsupervised-Learning-Ansatz (z. B. Clustering) werden auch Zielwerte (Targets) entfernt, damit das Modell rein auf Basis der Muster in den Daten lernt.

In [82]:
df = df.drop(columns=[
    # Data Leakage
    "impressions_s4",
    "days_after_start_s4",

    # Rohdaten
    "impressions_s1",
    "impressions_s2",
    "impressions_s3",
    "days_after_start_s1",
    "days_after_start_s2",
    "days_after_start_s3",
    "target",
    "remaining_target",
    "projected_total",

    # IDs & Zeitstempel
    "campaign_id",
    "start_week",
    "end_week",
    "end_month",
])

### 9. Kontrolle

In [83]:
df

,budget,start_month,days,region,category,facebook,google_display,google_search,instagram,needs_intervention,...,required_impr_per_day,growth_1_2,growth_2_3,growth_acceleration,projected_total,projected_ratio,speed_ratio,budget_per_day,n_channels,google_any
0,20400,9,57,germany,other,0,0,0,1,0,...,-389316.928571,1445833.0,1747435.0,301602.0,1.209253e+07,2.963856,-1.835105,357.894737,1,0
1,18300,12,154,austria,conference,1,1,1,1,0,...,4151.868421,675271.0,608187.0,-67084.0,4.163614e+06,1.137600,0.153566,118.831169,4,1
2,10900,5,33,switzerland,festival,1,1,1,1,0,...,46007.875000,334042.0,408846.0,74804.0,2.103997e+06,0.965136,0.721607,330.303030,4,1
3,10700,3,110,germany,conference,1,1,1,1,0,...,7526.464286,332777.0,373221.0,40444.0,2.300957e+06,1.075214,0.359812,97.272727,4,1
4,10600,5,473,germany,comedy,1,0,1,1,0,...,2056.983051,291655.0,430190.0,138535.0,2.218804e+06,1.046606,0.438503,22.410148,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,100,10,18,germany,conference,1,0,1,0,1,...,1415.750000,5182.0,4468.0,-714.0,1.586186e+04,0.793093,1.606590,5.555556,2,1
237,100,9,70,germany,comedy,1,0,0,1,0,...,441.333333,5874.0,1501.0,-4373.0,1.353692e+04,0.676846,2.282153,1.428571,2,0
238,100,12,21,switzerland,theatre,1,0,0,0,1,...,676.600000,1479.0,5006.0,3527.0,1.918481e+04,0.959241,0.740617,4.761905,1,0
239,100,4,29,germany,festival,1,0,0,0,0,...,395.000000,7527.0,3160.0,-4367.0,2.008250e+04,1.004125,0.570397,3.448276,1,0


In [84]:
df.to_csv("inputfiles/beab_datensatz.csv", index=False)